# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [10]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [11]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [12]:
# links = fetch_website_links("https://edwarddonner.com")
links = fetch_website_links("https://buildbharatmart.com")
links

['/products',
 '/',
 '/products',
 '/rentals',
 '/ans-services',
 '/cart',
 '/account',
 '/products',
 '/ans-services',
 '/products?category=POWER+TOOLS',
 '/products?category=HAND+TOOLS',
 '/products?category=PLUMBING',
 '/products?category=ELECTRICAL',
 '/products?category=PAINTING',
 '/products?category=SAFETY+GEAR',
 '/products',
 '/ans-services',
 '/rentals',
 '/products',
 '/signup',
 '/products',
 '/products?category=POWER+TOOLS',
 '/products?category=HAND+TOOLS',
 '/products?category=ELECTRICAL',
 '/products?category=PLUMBING',
 '/rentals',
 '/ans-services',
 '/about',
 '/faq',
 '/policies',
 '/policies',
 '/policies',
 '/policies/contact-us',
 'mailto:buildbharatmart@gmail.com',
 'tel:+919999998645',
 '/policies',
 '/about',
 '/policies/contact-us']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "products page", "url": "https://full.url/goes/here/products"},
        {"type": "rentals page", "url": "https://full.url/goes/here/rentals"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [15]:
# print(get_links_user_prompt("https://edwarddonner.com"))
print(get_links_user_prompt("https://buildbharatmart.com"))


Here is the list of links on the website https://buildbharatmart.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/products
/
/products
/rentals
/ans-services
/cart
/account
/products
/ans-services
/products?category=POWER+TOOLS
/products?category=HAND+TOOLS
/products?category=PLUMBING
/products?category=ELECTRICAL
/products?category=PAINTING
/products?category=SAFETY+GEAR
/products
/ans-services
/rentals
/products
/signup
/products
/products?category=POWER+TOOLS
/products?category=HAND+TOOLS
/products?category=ELECTRICAL
/products?category=PLUMBING
/rentals
/ans-services
/about
/faq
/policies
/policies
/policies
/policies/contact-us
mailto:buildbharatmart@gmail.com
tel:+919999998645
/policies
/about
/policies/contact-us


In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [17]:
select_relevant_links("https://buildbharatmart.com")

{'links': [{'type': 'about page', 'url': 'https://buildbharatmart.com/about'},
  {'type': 'services page', 'url': 'https://buildbharatmart.com/ans-services'},
  {'type': 'faq page', 'url': 'https://buildbharatmart.com/faq'},
  {'type': 'contact page',
   'url': 'https://buildbharatmart.com/policies/contact-us'}]}

In [18]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [19]:
# select_relevant_links("https://edwarddonner.com")
select_relevant_links("https://buildbharatmart.com")

Selecting relevant links for https://buildbharatmart.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'about page', 'url': 'https://buildbharatmart.com/about'},
  {'type': 'products page', 'url': 'https://buildbharatmart.com/products'},
  {'type': 'rentals page', 'url': 'https://buildbharatmart.com/rentals'},
  {'type': 'services page', 'url': 'https://buildbharatmart.com/ans-services'},
  {'type': 'faq page', 'url': 'https://buildbharatmart.com/faq'},
  {'type': 'contact page',
   'url': 'https://buildbharatmart.com/policies/contact-us'}]}

In [20]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Discord join page', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 21 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
6 days ago
•
1.37M
•
1.39k
zai-org/GLM-5.2
Updated
2 days ago
•
191k
•
3.35k
baidu/Unlimited-OCR
Updated
1 day ago
•
885k
•
1.7k
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
9 days ago
•
323k
•
691
deepseek-ai/DeepSeek-V4-Pro-DSpark
Updated


In [24]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""



In [ ]:
# # # Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n6 days ago\n•\n1.37M\n•\n1.39k\nzai-org/GLM-5.2\nUpdated\n2 days ago\n•

In [27]:
get_brochure_user_prompt("Build Bharat Mart", "https://buildbharatmart.com")

Selecting relevant links for https://buildbharatmart.com by calling gpt-5-nano
Found 7 relevant links


"\nYou are looking at a company called: Build Bharat Mart\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nBuild Bharat Mart - Premium Hardware, Tools & Home Improvement | India\n\n🚧\nWe're putting the finishing touches!\n— Some features are still under construction.\nLaunching fully very soon!\n🚀\nFree delivery on orders above ₹999 — PAN India shipping\nShop Now →\nBuild Bharat Mart\nPremium Hardware & Tools\nProducts\nProducts\nRentals\nRentals\nServices\nServices\nCart\nAccount\nTrusted by 5,000+ customers across India\nBuilding Your Dreams,\nOne Tool at a Time\nPremium hardware, tools, and expert services for professionals and DIY enthusiasts. Quality products at competitive prices with free delivery across India.\nShop Products\nExplore Services\nFree Delivery\nOn orders above ₹999 across India\nQuality Guaranteed\nPremium products from trust

In [28]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [29]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. It is a vibrant collaboration platform where machine learning enthusiasts, researchers, and professionals around the globe create, discover, and share machine learning models, datasets, and applications. The Hugging Face Hub empowers users to host and collaborate on unlimited public ML models and datasets, driving innovation in artificial intelligence.

---

## What We Offer

- **Models:** Access and contribute to over 2 million machine learning models spanning a diverse range of applications.
- **Datasets:** Explore and upload from a collection of over 500,000 datasets facilitating research and development.
- **Spaces:** Launch and interact with AI-powered applications running directly on the platform.
- **Buckets:** Secure cloud storage solutions designed for ML workflows.
- **HuggingChat:** Engage with cutting-edge conversational AI applications.
- **Enterprise Solutions:** Including Hugging Face PRO, Enterprise Support, Inference Providers, Endpoints, and Storage Buckets tailored to business needs.

---

## Community & Culture

At its core, Hugging Face fosters an open, collaborative AI community committed to democratizing machine learning technology. The company culture encourages:

- Open collaboration across developers, researchers, and enterprises.
- Inclusive community engagement via platforms like Discord, Forums, and GitHub.
- Sharing knowledge through Blogs, Daily Research Paper summaries, and Learning resources.
- Continuous innovation and transparency in AI development.

---

## Our Customers

Hugging Face serves a broad spectrum of customers, including:

- Independent AI researchers and model developers.
- Technology companies integrating AI into products.
- Enterprises requiring scalable AI infrastructure and expert support.
- Educational institutions leveraging datasets and models for learning.
- AI enthusiasts and hobbyists worldwide contributing to and benefiting from open-source AI.

---

## Careers & Opportunities

Hugging Face is growing rapidly and welcomes passionate individuals eager to impact the future of AI. Working here means joining a mission-driven team that values:

- Innovation and experimentation with the latest AI technology.
- Collaboration in a vibrant community.
- Opportunities to contribute to open-source projects used globally.
- Roles spanning research, engineering, community management, enterprise solutions, and more.

Check their website regularly for job postings and internship opportunities to join a team shaping the future of machine learning.

---

## Why Choose Hugging Face?

- Largest repository of open-source models and datasets.
- Seamless collaboration across the AI community.
- Robust tools and infrastructure for deploying and scaling AI applications.
- Strong support for enterprises looking to adopt AI responsibly.
- Active, friendly, and diverse community driving AI advancements.

---

## Connect with Hugging Face

- **Website:** https://huggingface.co
- **Community:** Discord, Forums, GitHub
- **Learn More:** Blogs, Daily Papers, Documentation

Join Hugging Face and be part of the AI revolution transforming industries and societies worldwide!

In [30]:
create_brochure("Build Bharat Mart", "https://buildbharatmart.com")

Selecting relevant links for https://buildbharatmart.com by calling gpt-5-nano
Found 6 relevant links


# Build Bharat Mart  
**Premium Hardware, Tools & Home Improvement | India**

---

### About Build Bharat Mart  
Founded in 2018 by the Solanki family, Build Bharat Mart began as a small neighborhood hardware shop in Delhi with a mission to make quality tools and materials accessible without overpaying or compromising service. Today, it is a trusted pan-India online platform serving over 5,000 happy customers with premium hardware, tools, rental options, and expert home improvement services.

Build Bharat Mart is more than a store — it’s a reliable partner for professionals and DIY enthusiasts alike, enabling every homeowner, contractor, and builder to bring their dreams to life with the right tools and expertise.

---

### Our Vision & Mission  

**Vision:**  
Become India’s most trusted and accessible home improvement platform — empowering all users with quality products, materials, and expert guidance.

**Mission:**  
Simplify home improvement by offering carefully curated, quality-assured products at fair prices, backed by knowledgeable support and reliable delivery, making project success effortless from start to finish.

---

### Products & Services  
Build Bharat Mart offers over **2,900 products** across **31 categories** from more than **20 trusted brands**, ensuring premium quality and wide selection:

- **Power Tools:** Drills, saws, grinders & more  
- **Hand Tools:** Hammers, wrenches, pliers & sets  
- **Plumbing:** Pipes, fittings, taps & fixtures  
- **Electrical:** Wiring, switches, panels & LEDs  
- **Painting:** Paints, brushes, rollers & primers  
- **Safety Gear:** Helmets, gloves, goggles & masks  
- **Tiles Collection & Building Materials**

**Additional Services:**  
- Tool Rentals: Carpentry, electrical, plumbing, painting  
- Professional Help: Verified experts for plumbing, electrical, carpentry, painting

---

### Why Choose Build Bharat Mart?  

- **Quality Guaranteed:** Premium products from trusted brands  
- **Free Delivery:** Pan India shipping on orders above ₹999  
- **Expert Support:** Knowledgeable, friendly staff to guide your project  
- **Secure Payments:** 100% safe transactions through Razorpay  
- **Trusted Partner:** Over 15,000 orders delivered and counting

---

### Company Culture  
Build Bharat Mart thrives on honesty, community, and a personal touch. From its origins as a family-run local store to a nationwide platform, the company fosters trust and values strong customer relationships. Empathy, expertise, and reliability shape the way Build Bharat Mart supports every project and customer.

---

### Customer Base  
Serving homeowners, contractors, builders, and DIYers across India, Build Bharat Mart is trusted by thousands who value quality, convenience, and expert advice in hardware and home improvement.

---

### Careers & Opportunities  
Join a growing team passionate about transforming India’s home improvement landscape. Whether you're skilled in logistics, customer support, technical expertise, or digital innovation, Build Bharat Mart offers a rewarding environment built on trust, growth, and community impact. Keep an eye on their careers page for upcoming roles as they expand.

---

### Connect & Shop  
Build Bharat Mart — Your trusted home improvement partner since 2018. Experience quality tools, rental options, and expert services with secure, reliable delivery across India.

**Website:** [Build Bharat Mart](#)  
**Shop Now:** Free delivery on orders above ₹999 — PAN India shipping  

---

*Building Your Dreams, One Tool at a Time.*  
© NNPYS Home Solutions LLP | 2018–2026

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [32]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


In [ ]:
stream_brochure("Build Bharat Mart", "https://buildbharatmart.com")

In [ ]:
# # Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Humorous Brochure for Build Bharat Mart:
stream_brochure("Build Bharat Mart", "https://buildbharatmart.com")

NameError: name 'stream_brochure' is not defined

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>